# Periodic GARCH: volatility with a calendar dummy

Bollerslev and Ghysels's (1996) periodic GARCH lets a GARCH(1,1) change with a calendar dummy s(t), such as a day after no trading: the mean, the level of the variance, and how much a shock feeds into the next day's variance can all differ when s(t) = 1. ARCHModels.jl fits GARCH(1,1), but no Julia package has a dummy in the variance equation, nor the periodic model. `pgarch` fits it by quasi-maximum likelihood with robust standard errors, with any of its parameters held at zero, and `diagnostics` gives the statistics of the paper's Table 2. They are tested against that table's GARCH(1,1) with a non-trading-day dummy and its P-GARCH(1,1), and against ARCHModels for the plain GARCH(1,1).

The data is `bg96`, from PollisDatasets: the 1,974 daily Deutsche mark / British pound returns of January 1984 to December 1991, with the paper's non-trading-day dummy.

In [ ]:
Pollis.packages("ARCHModels", "Optim", "ForwardDiff", "PollisDatasets", "Plots")

## Set-up

`replicate` prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share.

In [ ]:
using ARCHModels, Optim, ForwardDiff, PollisDatasets, Plots, Printf, Statistics, LinearAlgebra

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => ""))
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

## The method

The model of Bollerslev and Ghysels (1996, p. 146, equation 14 and the note to Table 2) has mean μ + μ₁ s(t) and conditional variance

σ²(t) = ω + ω₁ s(t) + (α + α₁ s(t)) (ε²(t−1) − ω − ω₁ s(t−1)) + β (σ²(t−1) − ω − ω₁ s(t−1)),

so ω + ω₁ s(t) is the level the variance returns to, ω₁ the extra variance when s(t) = 1, and α₁ how much less, or more, a shock on such a day feeds into the next day's variance. With α₁ = 0 it is GARCH(1,1) with a dummy, and with ω₁ = α₁ = μ₁ = 0 plain GARCH(1,1), whose ω is ARCHModels' intercept divided by 1 − α − β.

`pgarch` maximises the Gaussian likelihood and reports the robust standard errors of quasi-maximum likelihood, the sandwich of the Hessian and the outer product of the scores; the recursion starts from the sample variance of the returns, as ARCHModels does. `diagnostics` gives the statistics of Table 2: information criteria, larger being better; skewness and kurtosis of the standardized residuals; Ljung-Box statistics of them and of their squares for 20 lags; and the paper's three loss functions.

In [ ]:
const PARAMETERS = (:mu, :mu1, :omega, :omega1, :alpha, :alpha1, :beta)

"""
	pgarchvariances(θ, y, s)

Residuals and conditional variances of the periodic GARCH(1,1) for returns `y`, dummy `s` and θ = (μ, μ₁, ω, ω₁, α, α₁, β).
"""
function pgarchvariances(θ, y, s)
	μ, μ1, ω, ω1, α, α1, β = θ
	ε = y .- μ .- μ1 .* s
	h = similar(ε)
	h[1] = var(y)  # start from the sample variance
	for t in 2:length(y)
		level = ω + ω1 * s[t-1]  # the variance level of day t-1
		h[t] = ω + ω1 * s[t] + (α + α1 * s[t]) * (ε[t-1]^2 - level) + β * (h[t-1] - level)
	end
	ε, h
end

"""Gaussian log likelihood of each day, or -Inf everywhere if a variance is not positive."""
function pgarchloglik(θ, y, s)
	ε, h = pgarchvariances(θ, y, s)
	all(>(0), h) || return fill(oftype(first(h), -Inf), length(y))
	-(log(2pi) .+ log.(h) .+ ε .^ 2 ./ h) ./ 2
end

"""
	diagnostics(ε, h, logL, k)

The statistics of Bollerslev and Ghysels's Table 2 for residuals `ε`, variances `h`, log likelihood `logL` and `k` parameters:
AIC and SIC (larger is better), skewness b3 and kurtosis b4 of the standardized residuals, Ljung-Box Q and Q2 for 20 lags,
and the losses MSE, HMSE and LL.
"""
function diagnostics(ε, h, logL, k)
	T = length(ε)
	z = ε ./ sqrt.(h)
	ljungbox(x) = (c = x .- mean(x); T * (T + 2) * sum((sum(c[j+1:end] .* c[1:end-j]) / sum(abs2, c))^2 / (T - j) for j in 1:20))
	(AIC = 2logL - 2k, SIC = 2logL - k * log(T), b3 = mean(z .^ 3), b4 = mean(z .^ 4), Q = ljungbox(z), Q2 = ljungbox(z .^ 2),
		MSE = mean((ε .^ 2 .- h) .^ 2), HMSE = mean((ε .^ 2 ./ h .- 1) .^ 2), LL = mean(log.(ε .^ 2 ./ h) .^ 2))
end

"""
	pgarch(y, s; free=PARAMETERS, start)

Quasi-maximum likelihood fit of the periodic GARCH(1,1) to returns `y` with dummy `s`, the parameters not in `free` held at zero.
`start` gives the starting values of the free ones, in the order of `PARAMETERS`. Returns the estimates θ, robust standard errors,
conditional variances h, the log likelihood and the statistics of `diagnostics`.
"""
function pgarch(y, s; free=PARAMETERS, start)
	index = [findfirst(==(p), PARAMETERS) for p in free]
	expand(p) = (θ = zeros(eltype(p), 7); θ[index] = p; θ)
	nll(p) = -sum(pgarchloglik(expand(p), y, s))
	p = Optim.minimizer(optimize(nll, start, BFGS(), Optim.Options(g_tol=1e-10); autodiff=:forward))
	H = ForwardDiff.hessian(nll, p)
	G = ForwardDiff.jacobian(q -> pgarchloglik(expand(q), y, s), p)  # the score of each day
	se = sqrt.(diag(H \ (transpose(G) * G) / H))  # robust (sandwich) standard errors
	ε, h = pgarchvariances(expand(p), y, s)
	(θ = expand(p), se = expand(se), h = h, logL = -nll(p), diagnostics(ε, h, -nll(p), length(p))...)
end

## Tests

`table2` prints a column of Bollerslev and Ghysels's Table 2 (p. 146) beside Julia's: each estimate and robust standard error, then the statistics.

In [ ]:
bg = dataset("bg96");  # one row per trading day
y = Float64.(bg.return);  # daily percentage returns, 100 [ln(P_t) - ln(P_t-1)]
s = Float64.(bg.nontrading);  # 1 after a weekend or holiday, 0 otherwise
statistics = (:AIC, :SIC, :b3, :b4, :Q, :Q2, :MSE, :HMSE, :LL)

"""Replicate a column of Table 2: `published` holds each estimate with its standard error, and the statistics, as printed."""
function table2(title, m, published)
	rows = []
	for (i, name) in enumerate(PARAMETERS)
		haskey(published, name) || continue
		b, se = published[name]
		push!(rows, (string(name), m.θ[i], b), ("se " * string(name), m.se[i], se))
	end
	for name in statistics
		push!(rows, (string(name), m[name], published[name]))
	end
	replicate(title, rows)
end

### Plain GARCH(1,1): the same as ARCHModels

With the three dummy parameters held at zero, `pgarch` is GARCH(1,1), and must give ARCHModels' estimates, robust standard errors and log likelihood. ARCHModels' intercept is converted to the paper's ω, with a delta-method standard error.

In [ ]:
am = fit(GARCH{1, 1}, y; meanspec=Intercept)  # robust standard errors by default
w, b1, a1, mu = coef(am)  # intercept, beta, alpha, mean
level = w / (1 - a1 - b1)
grad = [1, level, level, 0] ./ (1 - a1 - b1)
se_am = sqrt.(diag(vcov(am)))
m4 = pgarch(y, s; free=(:mu, :omega, :alpha, :beta), start=[0.0, 0.25, 0.15, 0.8])
@printf("%-14s%16s%16s%12s\n", "", "pgarch", "ARCHModels", "difference")
for (label, a, b) in (("mu", m4.θ[1], mu), ("omega", m4.θ[3], level), ("alpha", m4.θ[5], a1), ("beta", m4.θ[7], b1),
		("se mu", m4.se[1], se_am[4]), ("se omega", m4.se[3], sqrt(transpose(grad) * vcov(am) * grad)), ("se alpha", m4.se[5], se_am[3]),
		("se beta", m4.se[7], se_am[2]), ("log lik.", m4.logL, loglikelihood(am)))
	@printf("%-14s%16.8f%16.8f%12.1e\n", label, a, b, a - b)
end

### GARCH(1,1) with the dummy, and P-GARCH(1,1)

The fifth column of Table 2 adds ω₁ to GARCH(1,1); the sixth is P-GARCH(1,1), with α₁ free as well. With it, shocks on ordinary days persist (α + β about 1.000), while those after no trading die out faster (α + α₁ + β = .889) (p. 146).

In [ ]:
m5 = pgarch(y, s; free=(:mu, :omega, :omega1, :alpha, :beta), start=[0.0, 0.25, 0.05, 0.15, 0.8])
table2("GARCH(1,1) with the dummy (Bollerslev and Ghysels 1996, Table 2, column 5)", m5, (mu = ("-.009", ".008"), omega = (".259", ".086"),
	omega1 = (".054", ".022"), alpha = (".137", ".035"), beta = (".832", ".044"),
	AIC = "-2,191.0", SIC = "-2,219.0", b3 = "-.44", b4 = "5.89", Q = "18.8", Q2 = "21.7", MSE = ".252", HMSE = "4.88", LL = "8.21"))

m6 = pgarch(y, s; free=(:mu, :omega, :omega1, :alpha, :alpha1, :beta), start=[0.0, 0.3, 0.05, 0.15, -0.1, 0.8])
table2("P-GARCH(1,1) (Table 2, column 6)", m6, (mu = ("-.006", ".008"), omega = (".341", ".169"), omega1 = (".043", ".026"), alpha = (".178", ".043"),
	alpha1 = ("-.111", ".044"), beta = (".822", ".042"),
	AIC = "-2,179.3", SIC = "-2,212.9", b3 = "-.46", b4 = "5.87", Q = "19.1", Q2 = "23.9", MSE = ".257", HMSE = "4.67", LL = "8.66"))
replicate("Persistence of a shock (p. 146)", [("alpha + beta", m6.θ[5] + m6.θ[7], "1.000"), ("alpha + alpha1 + beta", m6.θ[5] + m6.θ[6] + m6.θ[7], ".889")])

## Example

The conditional volatility of the two models over a stretch of 1989-1990. The periodic model lets a Monday's shock feed less into Tuesday's variance, so its volatility jumps less after weekends.

In [ ]:
days = 1400:1600
plot(days, sqrt.(m5.h[days]), label="GARCH(1,1) with the dummy", xlabel="trading day", ylabel="conditional volatility (%)", size=(560, 330))
plot!(days, sqrt.(m6.h[days]), label="P-GARCH(1,1)")

## What differs

Plain GARCH(1,1): the estimates, standard errors and log likelihood agree with ARCHModels.jl to about 1e-13.

Table 2, column 5: all 19 values agree with the published ones to the printed digits except three last-digit differences: se of the dummy .023 vs .022, b3 -.43 vs -.44 and b4 5.88 vs 5.89.

Table 2, column 6 (P-GARCH): every coefficient agrees to 3 decimals. The standard errors of omega, omega1 and alpha differ by one in the third decimal (.171 vs .169, .027 vs .026, .042 vs .043), and SIC by one in the last digit (-2212.8 vs -2212.9). These are numerical-Hessian differences.

The one gap not explained is the skewness and kurtosis of column 6: -.43 and 5.63 here against -.46 and 5.87 in the paper, while the Q statistics, MSE, HMSE and LL of the same standardized residuals all agree. The paper may compute b3 and b4 on a different sample or residual definition.

Persistence: alpha + alpha1 + beta = .889 agrees; alpha + beta = 1.0006 rounds to 1.001 against the paper's 1.000.

## References

- Bollerslev, T. and Ghysels, E. (1996). Periodic autoregressive conditional heteroscedasticity. *Journal of Business & Economic Statistics* 14(2), 139-151. doi:10.1080/07350015.1996.10524640
- Ljung, G. M. and Box, G. E. P. (1978). On a measure of lack of fit in time series models. *Biometrika* 65(2), 297-303.
- Data: the returns as distributed with ARCHModels.jl, and the non-trading-day dummy as distributed with the R package rugarch (`dmbp`).